In [1]:
%pip install dotenv
%pip install pypdf2
%pip install pdfrw
%pip install reportlab
%pip install --upgrade pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restar

In [1]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import StringIO

# Load environment variables from environment.env
load_dotenv("environment.env")

# Get values from env
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Initialize client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(template_container_name)

# Example: List blobs
print("Blobs in container:")
for blob in container_client.list_blobs():
    print("  -", blob.name)


Blobs in container:
  - ADA Notice to Borrowers-FORM.pdf
  - Acknowledgement of Receipt - FORM.pdf
  - Agreement to Cooperate-FORM.pdf
  - Condominium Rider-FORM.pdf
  - Disclosure Statement-FORM.pdf
  - Escrow Waiver Affidavit-FORM.pdf
  - Info Disclosure Author-FORM.pdf
  - Mortgage-FORM.pdf
  - Perjury Statement-FORM.pdf
  - Primary Residence Affidavit-FORM.pdf
  - Promissory Note-FORM.pdf
  - SSICollect-FORM.pdf


In [3]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader
from io import BytesIO

# Load env file
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Connect to blob
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(template_container_name)

# Iterate through all PDFs in the container
for blob in container_client.list_blobs():
    if blob.name.lower().endswith(".pdf"):
        print(f"\n📄 Processing PDF: {blob.name}")
        
        # Download PDF into memory
        blob_client = container_client.get_blob_client(blob.name)
        pdf_bytes = blob_client.download_blob().readall()
        pdf_stream = BytesIO(pdf_bytes)

        # Read PDF
        reader = PdfReader(pdf_stream)

        # Embark on reading page by page
        for i, page in enumerate(reader.pages):
            print(f"\n--- Page {i+1} ---")
        
            if "/Annots" in page:
                for annot in page["/Annots"]:
                    annot_obj = annot.get_object()
                    if annot_obj.get("/Subtype") == "/Widget":  # Widget = form field
                        field = annot_obj.get("/T")   # Field name
                        value = annot_obj.get("/V")   # Field value
                        print(f"Field: {field}, Value: {value}")
            else:
                print("No form fields on this page.")
                
            # Extract form fields
            #    fields = page.get_fields()
            #    if fields:
            #        print("   Fillable fields:")
            #        for field_name, field_info in fields.items():
            #            field_type = field_info.get("/FT")  # e.g. /Tx, /Btn, /Ch
            #            field_value = field_info.get("/V")
            #            print(f"    - {field_name} : {field_type} (current value: {field_value})")
            #    else:
            #        print("   ⚠️ No fillable fields found.")



📄 Processing PDF: ADA Notice to Borrowers-FORM.pdf

--- Page 1 ---
Field: BORROWER, Value: None
Field: COBORROWER, Value: None
Field: DATE, Value: None
Field: DATE_2, Value: None

📄 Processing PDF: Acknowledgement of Receipt - FORM.pdf

--- Page 1 ---
Field: borrowers_line, Value: None
Field: Date, Value: None
Field: Loan Number, Value: None
Field: Property Address, Value: None
Field: Picture ID for All Borrowers, Value: None
Field: Promissory Note, Value: None
Field: Mortgage, Value: None
Field: Condo Rider, Value: None
Field: TruthInLending Disclosure Statement, Value: None
Field: Right of Rescission 2 copies for the borrower, Value: None
Field: Escrow Disclosure Loan Proceeds to Be Put in Escrow, Value: None
Field: Primary Residence Affidavit, Value: None
Field: Agreement to Cooperate, Value: None
Field: Disclosure Statement, Value: None
Field: Information Authorization Disclosure, Value: None
Field: Perjury Statement, Value: None
Field: SSN Policy, Value: None
Field: ADA Notice to

In [3]:
import os
import json
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from io import StringIO
import pandas as pd

# Load env file
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")

# Initialize client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = sample_container_name
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"

# Derive JSON blob name by replacing extension
json_blob_name = os.path.splitext(csv_blob_name)[0] + ".json"

# Get blob client for CSV
blob_client = blob_service_client.get_blob_client(container=container_name, blob=csv_blob_name)

# Download file content
stream = blob_client.download_blob()
data = stream.readall()

# Convert to pandas DataFrame
df = pd.read_csv(StringIO(data.decode("utf-8")))

# Convert DataFrame to JSON (row-oriented)
json_records = df.to_json(orient="records", indent=4)

# Upload JSON to blob storage
json_blob_client = blob_service_client.get_blob_client(container=container_name, blob=json_blob_name)
json_blob_client.upload_blob(json_records, overwrite=True)

print(f"✅ JSON file saved to container '{container_name}' as '{json_blob_name}'")

✅ JSON file saved to container 'sample' as 'Promissory_Note_Dummy_Data__50_rows_.json'


In [9]:
import os
import json
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.generic import NameObject, BooleanObject
import pandas as pd
from io import BytesIO, StringIO

# Load environment variables
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Container names
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled"            # output PDFs

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Step 1: Load CSV from blob into DataFrame ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)

stream = blob_client.download_blob()
data = stream.readall()

df = pd.read_csv(StringIO(data.decode("utf-8")))

# --- Step 2: Process each row in the DataFrame ---
for _, row in df.iterrows():
    loan_number = str(row["Loan Number"])  # Use Loan Number as filename
    print(f"\n📝 Filling Promissory Note for Loan {loan_number}")

    # Download PDF template
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()

    # Fill the PDF
    reader = PdfReader(BytesIO(template_pdf_data))
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)

    # Convert row into dict {field_name: value}
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}

    # --- Step 3: Update fields on ALL pages ---
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields_to_fill)

    # Save filled PDF to memory
    output_stream = BytesIO()
    writer.write(output_stream)
    output_stream.seek(0)

    # Upload to output container
    output_blob_name = f"{loan_number}.pdf"
    output_blob_client = blob_service_client.get_blob_client(
        container=output_container_name,
        blob=output_blob_name
    )
    output_blob_client.upload_blob(output_stream, overwrite=True)


    print(f"✅ Uploaded filled PDF as {output_container_name}/{output_blob_name}")



📝 Filling Promissory Note for Loan LN-100000
✅ Uploaded filled PDF as promissory-note-filled/LN-100000.doc

📝 Filling Promissory Note for Loan LN-100001
✅ Uploaded filled PDF as promissory-note-filled/LN-100001.doc

📝 Filling Promissory Note for Loan LN-100002
✅ Uploaded filled PDF as promissory-note-filled/LN-100002.doc

📝 Filling Promissory Note for Loan LN-100003
✅ Uploaded filled PDF as promissory-note-filled/LN-100003.doc

📝 Filling Promissory Note for Loan LN-100004
✅ Uploaded filled PDF as promissory-note-filled/LN-100004.doc

📝 Filling Promissory Note for Loan LN-100005
✅ Uploaded filled PDF as promissory-note-filled/LN-100005.doc

📝 Filling Promissory Note for Loan LN-100006
✅ Uploaded filled PDF as promissory-note-filled/LN-100006.doc

📝 Filling Promissory Note for Loan LN-100007
✅ Uploaded filled PDF as promissory-note-filled/LN-100007.doc

📝 Filling Promissory Note for Loan LN-100008
✅ Uploaded filled PDF as promissory-note-filled/LN-100008.doc

📝 Filling Promissory Note f

In [11]:
%pip install PyPDF2


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.generic import NameObject
from io import BytesIO
import pandas as pd
import pikepdf
 
# Load environment variables
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Container names
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled"            # output PDFs

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Step 1: Load CSV from blob into DataFrame ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)

stream = blob_client.download_blob()
data = stream.readall()

df = pd.read_csv(StringIO(data.decode("utf-8")))


# --- Step 2: Process each row in the DataFrame ---
for _, row in df.iterrows():
    loan_number = str(row["Loan Number"])  # Use Loan Number as filename
    print(f"\n📝 Filling Promissory Note for Loan {loan_number}")

    # Download PDF template
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()

    # Fill the PDF
    reader = PdfReader(BytesIO(template_pdf_data))
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)
 
    # Copy pages
    for page in reader.pages:
        writer.add_page(page)
 
    # Prepare fields to fill
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}
 
    # Fill fields on all pages
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields_to_fill)
 
    # Save intermediate filled PDF to memory
    filled_pdf_stream = BytesIO()
    writer.write(filled_pdf_stream)
    filled_pdf_stream.seek(0)
 
    # --- Step 3: Flatten PDF using pikepdf ---
    with pikepdf.open(filled_pdf_stream) as pdf:
        for page in pdf.pages:
            if "/Annots" in page:
                del page["/Annots"]

        # Safely access Root
        root = pdf.trailer.get("/Root")
        if root and "/AcroForm" in root:
            del root["/AcroForm"]
 
        # Save flattened PDF to memory
        flattened_pdf_stream = BytesIO()
        pdf.save(flattened_pdf_stream)
        flattened_pdf_stream.seek(0)
 
    # --- Step 4: Upload to Azure Blob Storage ---
    from azure.storage.blob import BlobServiceClient
 
    # Example: initialize blob service client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    output_container_name = output_container_name
    output_blob_name = f"{loan_number}.pdf"
 
    output_blob_client = blob_service_client.get_blob_client(
        container=output_container_name,
        blob=output_blob_name
    )
 
    output_blob_client.upload_blob(flattened_pdf_stream, overwrite=True)
    print(f"✅ Uploaded flattened PDF as {output_container_name}/{output_blob_name}")


📝 Filling Promissory Note for Loan LN-100000
✅ Uploaded flattened PDF as promissory-note-filled/LN-100000.pdf

📝 Filling Promissory Note for Loan LN-100001
✅ Uploaded flattened PDF as promissory-note-filled/LN-100001.pdf

📝 Filling Promissory Note for Loan LN-100002
✅ Uploaded flattened PDF as promissory-note-filled/LN-100002.pdf

📝 Filling Promissory Note for Loan LN-100003
✅ Uploaded flattened PDF as promissory-note-filled/LN-100003.pdf

📝 Filling Promissory Note for Loan LN-100004
✅ Uploaded flattened PDF as promissory-note-filled/LN-100004.pdf

📝 Filling Promissory Note for Loan LN-100005
✅ Uploaded flattened PDF as promissory-note-filled/LN-100005.pdf

📝 Filling Promissory Note for Loan LN-100006
✅ Uploaded flattened PDF as promissory-note-filled/LN-100006.pdf

📝 Filling Promissory Note for Loan LN-100007
✅ Uploaded flattened PDF as promissory-note-filled/LN-100007.pdf

📝 Filling Promissory Note for Loan LN-100008
✅ Uploaded flattened PDF as promissory-note-filled/LN-100008.pdf



In [18]:
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.generic import NameObject, BooleanObject
from io import BytesIO
import pandas as pd
import pikepdf

 
# Load environment variables
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Container names
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled"            # output PDFs

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Step 1: Load CSV from blob into DataFrame ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)

stream = blob_client.download_blob()
data = stream.readall()

df = pd.read_csv(StringIO(data.decode("utf-8")))


# --- Step 2: Process each row in the DataFrame ---
for _, row in df.iterrows():
    loan_number = str(row["Loan Number"])  # Use Loan Number as filename
    print(f"\n📝 Filling Promissory Note for Loan {loan_number}")

    # Download PDF template
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()

    # Fill the PDF
    reader = PdfReader(BytesIO(template_pdf_data))
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)
 
    # Copy pages
    for page in reader.pages:
        writer.add_page(page)
 
    # Prepare fields to fill
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}
 
    # Fill fields on all pages
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields_to_fill)
 
    # Force appearances so values render (this makes them persist)
    if "/AcroForm" in writer._root_object:
        writer._root_object["/AcroForm"].update({
            NameObject("/NeedAppearances"): BooleanObject(True)
     })

     # Save flattened PDF
    output_stream = BytesIO()
    writer.write(output_stream)
    output_stream.seek(0)
  
    # --- Step 4: Upload to Azure Blob Storage ---
    from azure.storage.blob import BlobServiceClient
 
    # Example: initialize blob service client
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    output_container_name = output_container_name
    output_blob_name = f"{loan_number}.pdf"
 
    output_blob_client = blob_service_client.get_blob_client(
        container=output_container_name,
        blob=output_blob_name
    )
 
    output_blob_client.upload_blob(output_stream, overwrite=True)
    print(f"✅ Uploaded flattened PDF as {output_container_name}/{output_blob_name}")


📝 Filling Promissory Note for Loan LN-100000
✅ Uploaded flattened PDF as promissory-note-filled/LN-100000.pdf

📝 Filling Promissory Note for Loan LN-100001
✅ Uploaded flattened PDF as promissory-note-filled/LN-100001.pdf

📝 Filling Promissory Note for Loan LN-100002
✅ Uploaded flattened PDF as promissory-note-filled/LN-100002.pdf

📝 Filling Promissory Note for Loan LN-100003
✅ Uploaded flattened PDF as promissory-note-filled/LN-100003.pdf

📝 Filling Promissory Note for Loan LN-100004
✅ Uploaded flattened PDF as promissory-note-filled/LN-100004.pdf

📝 Filling Promissory Note for Loan LN-100005
✅ Uploaded flattened PDF as promissory-note-filled/LN-100005.pdf

📝 Filling Promissory Note for Loan LN-100006
✅ Uploaded flattened PDF as promissory-note-filled/LN-100006.pdf

📝 Filling Promissory Note for Loan LN-100007
✅ Uploaded flattened PDF as promissory-note-filled/LN-100007.pdf

📝 Filling Promissory Note for Loan LN-100008
✅ Uploaded flattened PDF as promissory-note-filled/LN-100008.pdf



In [6]:
import os
import json
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.generic import NameObject, BooleanObject
import pandas as pd
from io import BytesIO, StringIO
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from PyPDF2 import PdfReader as RLReader

# Load environment variables
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Container names
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled"           # output PDFs

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Step 1: Load CSV from blob into DataFrame ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)

stream = blob_client.download_blob()
data = stream.readall()

df = pd.read_csv(StringIO(data.decode("utf-8")))

# --- Step 2: Process each row in the DataFrame ---
for _, row in df.iterrows():
    loan_number = str(row["Loan Number"])  # Use Loan Number as filename
    print(f"\n📝 Filling Promissory Note for Loan {loan_number}")

    # Download PDF template
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()

    # Fill the PDF with field values
    reader = PdfReader(BytesIO(template_pdf_data))
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)

    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}

    # Create overlay PDF with text drawn on coordinates
    packet = BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)

    for page_num, page in enumerate(reader.pages):
        if "/Annots" in page:
            for annot in page["/Annots"]:
                field = annot.get_object()
                field_name = field.get("/T")
                if field_name and field_name in fields_to_fill:
                    value = fields_to_fill[field_name]
                    rect = field.get("/Rect")
                    if rect and len(rect) == 4:
                        # Convert Decimal → float
                        x1, y1, x2, y2 = [float(v) for v in rect]

                        # Debug logging
                        print(f"   ➡ Field: {field_name}, Value: '{value}', Coords: ({x1}, {y1}, {x2}, {y2})")

                        # Draw text slightly inset
                        c.drawString(x1 + 2, y1 + 2, value)

    c.save()

    # Move overlay back to start
    packet.seek(0)
    overlay_pdf = RLReader(packet)

    # Merge overlay into original PDF
    for i in range(len(writer.pages)):
        writer.pages[i].merge_page(overlay_pdf.pages[0])  # Overlay text

    # Save filled (flattened) PDF
    output_stream = BytesIO()
    writer.write(output_stream)
    output_stream.seek(0)

    # Upload to output container
    output_blob_name = f"{loan_number}.pdf"
    output_blob_client = blob_service_client.get_blob_client(
        container=output_container_name,
        blob=output_blob_name
    )
    output_blob_client.upload_blob(output_stream, overwrite=True)

    print(f"✅ Uploaded filled + flattened PDF as {output_container_name}/{output_blob_name}")



📝 Filling Promissory Note for Loan LN-100000


In [12]:
import os
import pandas as pd
from io import BytesIO
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader, PdfWriter

# --- Azure setup ---
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")
output_container_name = os.getenv("OUTPUT_CONTAINER_NAME")
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Load CSV from blob into pandas ---
csv_blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob="Promissory_Note_Dummy_Data__50_rows_.csv")
csv_bytes = csv_blob_client.download_blob().readall()
df = pd.read_csv(BytesIO(csv_bytes))

# --- Fill PDF for each row in DataFrame ---
for idx, row in df.iterrows():
    fields_to_fill = row.to_dict()  # convert row to dict
    pdf_name = "promissory-note-form.pdf"
    print(f"📝 Filling PDF for row {idx+1}: {pdf_name}")

    # Download template PDF
    template_blob_client = blob_service_client.get_blob_client(container=template_container_name, blob=pdf_name)
    pdf_bytes = template_blob_client.download_blob().readall()
    pdf_stream = BytesIO(pdf_bytes)

    # Read and clone PDF
    reader = PdfReader(pdf_stream)
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)

    # Fill fields
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields_to_fill)

    # Force appearance update
    root = writer._root.get_object()
    acro_form = root.get("/AcroForm")
    if acro_form:
        acro_obj = acro_form.get_object()
        for f in acro_obj.get("/Fields", []):
            f_obj = f.get_object()
            field_name = f_obj.get("/T")
            if field_name in fields_to_fill:
                f_obj.update({
                    "/V": str(fields_to_fill[field_name]),  # set value
                    "/Ff": 0  # visible & editable
                })

    # Save filled PDF to memory
    output_stream = BytesIO()
    writer.write(output_stream)
    output_stream.seek(0)

    # Upload to output container
    output_blob_client = blob_service_client.get_blob_client(container=output_container_name, blob=f"filled_promissory_note_{idx+1}.pdf")
    output_blob_client.upload_blob(output_stream, overwrite=True)

    print(f"✅ Uploaded filled PDF for row {idx+1}")


📝 Filling PDF for row 1: promissory-note-form.pdf


ResourceNotFoundError: The specified blob does not exist.
RequestId:343c0354-001e-0056-803f-1ce415000000
Time:2025-09-02T19:23:13.1663556Z
ErrorCode:BlobNotFound
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>BlobNotFound</Code><Message>The specified blob does not exist.
RequestId:343c0354-001e-0056-803f-1ce415000000
Time:2025-09-02T19:23:13.1663556Z</Message></Error>

In [4]:
import os
import re
import json
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader
from io import BytesIO

# Load env file
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Connect to blob service
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
template_container_client = blob_service_client.get_container_client(template_container_name)

# Helper: sanitize blob names for JSON keys
def sanitize_name(name: str) -> str:
    # Lowercase
    name = name.lower()
    # Replace spaces with hyphens
    name = name.replace(" ", "-")
    # Replace invalid characters with hyphens
    name = re.sub(r"[^a-z0-9.\-]", "-", name)
    return name

# Result dict
all_pdfs_fields = {}

# Iterate through all PDFs in the container
for blob in template_container_client.list_blobs():
    if blob.name.lower().endswith(".pdf"):
        print(f"\n📄 Processing PDF: {blob.name}")

        # Download PDF into memory
        blob_client = template_container_client.get_blob_client(blob.name)
        pdf_bytes = blob_client.download_blob().readall()
        pdf_stream = BytesIO(pdf_bytes)

        # Read PDF
        reader = PdfReader(pdf_stream)

        # Extract form fields
        fields = reader.get_fields()
        pdf_fields = []
        if fields:
            for field_name, field_info in fields.items():
                field_type = field_info.get("/FT")  # /Tx, /Btn, /Ch, /Sig
                field_value = field_info.get("/V")
                pdf_fields.append({
                    "field_name": field_name,
                    "field_type": field_type,
                    "current_value": field_value
                })

        # Use sanitized blob name as JSON key
        sanitized_name = sanitize_name(blob.name)
        all_pdfs_fields[sanitized_name] = pdf_fields

# Convert to JSON
json_data = json.dumps(all_pdfs_fields, indent=4)

# Upload JSON to "fields-json" container
output_container_name = "fields-json"   # ✅ valid container name
output_blob_name = "fields.json"

# Ensure container exists (create if not present)
output_container_client = blob_service_client.get_container_client(output_container_name)
try:
    output_container_client.create_container()
except Exception:
    pass  # Ignore if it already exists

# Upload the JSON file
output_blob_client = output_container_client.get_blob_client(output_blob_name)
output_blob_client.upload_blob(json_data, overwrite=True)

print(f"\n✅ JSON uploaded to container '{output_container_name}' as '{output_blob_name}'")



📄 Processing PDF: ADA Notice to Borrowers-FORM.pdf

📄 Processing PDF: Acknowledgement of Receipt - FORM.pdf

📄 Processing PDF: Agreement to Cooperate-FORM.pdf

📄 Processing PDF: Condominium Rider-FORM.pdf

📄 Processing PDF: Disclosure Statement-FORM.pdf

📄 Processing PDF: Escrow Waiver Affidavit-FORM.pdf

📄 Processing PDF: Info Disclosure Author-FORM.pdf

📄 Processing PDF: Mortgage-FORM.pdf

📄 Processing PDF: Perjury Statement-FORM.pdf

📄 Processing PDF: Primary Residence Affidavit-FORM.pdf

📄 Processing PDF: Promissory Note-FORM.pdf

📄 Processing PDF: SSICollect-FORM.pdf

✅ JSON uploaded to container 'fields-json' as 'fields2.json'


In [5]:
import os
import json
from io import BytesIO
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader, PdfWriter

# Load environment variables
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")
data_container_name = os.getenv("DATA_CONTAINER_NAME")
output_container_name = os.getenv("OUTPUT_CONTAINER_NAME")

# Connect to blob
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
template_container = blob_service_client.get_container_client(template_container_name)
data_container = blob_service_client.get_container_client(data_container_name)
output_container = blob_service_client.get_container_client(output_container_name)

# Load JSON from blob
json_blob_client = data_container.get_blob_client("LN-123456/LN-123456.json")
json_bytes = json_blob_client.download_blob().readall()
pdf_data = json.loads(json_bytes)

# Function to normalize strings (lowercase, remove punctuation, spaces)
def normalize_name(name):
    return ''.join(e.lower() for e in name if e.isalnum())

# Iterate through all PDFs
for blob in template_container.list_blobs():
    if blob.name.lower().endswith(".pdf"):
        print(f"📝 Filling PDF: {blob.name}")

        # Find matching JSON key
        normalized_blob_name = normalize_name(blob.name)
        matched_key = None
        for key in pdf_data.keys():
            if normalize_name(key) == normalized_blob_name:
                matched_key = key
                break

        if not matched_key:
            print(f"⚠️ No data found in JSON for {blob.name}, skipping...")
            continue

        fields_to_fill = {}
        for field in pdf_data[matched_key]["fields"]:
            fields_to_fill[field["id"]] = field.get("value", "Sample Data")  # fallback value

        # Download PDF into memory
        blob_client = template_container.get_blob_client(blob.name)
        pdf_bytes = blob_client.download_blob().readall()
        reader = PdfReader(BytesIO(pdf_bytes))
        writer = PdfWriter()

        # Copy pages and fill fields
        for page in reader.pages:
            writer.add_page(page)
            writer.update_page_form_field_values(page, fields_to_fill)

        # Save to memory
        output_stream = BytesIO()
        writer.write(output_stream)
        output_stream.seek(0)

        # Upload to output container
        output_blob_client = output_container.get_blob_client(blob.name)
        output_blob_client.upload_blob(output_stream, overwrite=True)
        print(f"✅ Uploaded filled PDF to {output_container_name}/{blob.name}")


📝 Filling PDF: ADA Notice to Borrowers-FORM.pdf
⚠️ No data found in JSON for ADA Notice to Borrowers-FORM.pdf, skipping...
📝 Filling PDF: Acknowledgement of Receipt - FORM.pdf
⚠️ No data found in JSON for Acknowledgement of Receipt - FORM.pdf, skipping...
📝 Filling PDF: Agreement to Cooperate-FORM.pdf
⚠️ No data found in JSON for Agreement to Cooperate-FORM.pdf, skipping...
📝 Filling PDF: Condominium Rider-FORM.pdf
⚠️ No data found in JSON for Condominium Rider-FORM.pdf, skipping...
📝 Filling PDF: Disclosure Statement-FORM.pdf
⚠️ No data found in JSON for Disclosure Statement-FORM.pdf, skipping...
📝 Filling PDF: Escrow Waiver Affidavit-FORM.pdf
⚠️ No data found in JSON for Escrow Waiver Affidavit-FORM.pdf, skipping...
📝 Filling PDF: Info Disclosure Author-FORM.pdf
⚠️ No data found in JSON for Info Disclosure Author-FORM.pdf, skipping...
📝 Filling PDF: Mortgage-FORM.pdf
⚠️ No data found in JSON for Mortgage-FORM.pdf, skipping...
📝 Filling PDF: Perjury Statement-FORM.pdf
⚠️ No data found